# Patrón Creacional: Factory Method

Siguiendo con el ejemplo del restaurante utilizado en las actividades pasadas, supongamos que dicho establecimiento maneja diferentes tipos de pedidos como:

* Pedido para consumir en el restaurante.
* Pedido a domicilio.

## Sin patrón Factory Method ❌

Si no se aplica el patrón, el problema aparece cuando el código que utiliza los pedidos debe conocer todas las clases concretas y decidir mediante `if/elif` cuál debe crear. Como consecuencia, si aparecen nuevos tipos, el código principal deberá modificarse. Esto hace que la creación de pedidos sea cada vez más difícil de mantener.

In [6]:
from abc import ABC, abstractmethod

class Order(ABC):
    def __init__(self, order_id: str, total: float) -> None:
        self.order_id = order_id
        self.total = total

    @abstractmethod
    def prepare(self) -> str:
        pass

    @abstractmethod
    def summary(self) -> str:
        pass

class DineInOrder(Order):
    def __init__(self, order_id: str, total: float, table: int) -> None:
        super().__init__(order_id, total)
        self.table = table

    def prepare(self) -> str:
        return f"Preparando pedido {self.order_id} para la mesa {self.table}"

    def summary(self) -> str:
        return f"Pedido {self.order_id} - Mesa {self.table} - ${self.total:.2f}"

class DeliveryOrder(Order):
    def __init__(self, order_id: str, total: float, address: str ) -> None:
        super().__init__(order_id, total)
        self.address = address

    def prepare(self) -> str:
        return f"Preparando pedido {self.order_id} para domicilio"

    def summary(self) -> str:
        return f"Pedido {self.order_id} - Dirección: {self.address} - ${self.total:.2f}"


tipo_pedido = "domicilio"

if tipo_pedido == "salon":
    pedido = DineInOrder("001", 45000, 5)
elif tipo_pedido == "domicilio":
    pedido = DeliveryOrder("002", 52000, "Calle 20 #15-30")

print(pedido.prepare())
print(pedido.summary())


Preparando pedido 002 para domicilio
Pedido 002 - Dirección: Calle 20 #15-30 - $52000.00


## Con patrón Factory Method ✅
Aplicando el patrón, se traslada la responsabilidad de creación a una jerarquía de creadores. En este caso, la clase padre Restaurant es la que define el __Factory Method__ con el método de crear pedido y a su vez, dos clases creadoras que se encargan de fabricar cada tipo de pedido respectivamente. De esta forma, el método de crear pedido no necesita conocer directamente las clases específicas.

In [10]:
class Restaurant(ABC):
    def __init__(self, name: str, city: str) -> None:
        self.name = name
        self.city = city

    @abstractmethod
    def create_order(self, number: str, total: float) -> Order:
        pass

    def take_order(self, number: str, total: float) -> None:
        order = self.create_order(number, total)

        print(order.prepare())
        print(order.summary())


class DineInRestaurant(Restaurant):
    def __init__(self, name: str, city: str, table: int) -> None:
        super().__init__(name, city)
        self.table = table

    def create_order(self, number: str, total: float) -> Order:
        return DineInOrder(number, total, self.table)


class DeliveryRestaurant(Restaurant):
    def __init__(self, name: str, city: str, address: str) -> None:
        super().__init__(name, city)
        self.address = address

    def create_order(self, number: str, total: float) -> Order:
        return DeliveryOrder(number, total, self.address)


dine_in_restaurant = DineInRestaurant("Pasta Palace", "Santa Marta", 5)
delivery_restaurant = DeliveryRestaurant("Pasta Palace", "Santa Marta", "Calle 20 #15-30")

dine_in_restaurant.take_order("001", 45000)
delivery_restaurant.take_order("002", 52000)


Preparando pedido 001 para la mesa 5
Pedido 001 - Mesa 5 - $45000.00
Preparando pedido 002 para domicilio
Pedido 002 - Dirección: Calle 20 #15-30 - $52000.00


## Diagrama UML
```plantuml
@startuml
abstract class Order {
    - number: str
    - total: float
    + prepare()
    + summary()
}
class DineInOrder {
    - table: int
    + prepare()
    + summary()
}
class DeliveryOrder {
    - address: str
    + prepare()
    + summary()
}
abstract class Restaurant {
    - name: str
    - city: str
    + create_order()
    + take_order()
}
class DineInRestaurant {
    - table: int
    + create_order()
}
class DeliveryRestaurant {
    - address: str
    + create_order()
}
Order <|-- DineInOrder
Order <|-- DeliveryOrder
Restaurant <|-- DineInRestaurant
Restaurant <|-- DeliveryRestaurant
Restaurant ..> Order : creates
@enduml
```

https://www.plantuml.com/plantuml/duml/bP71IiGm48RlUOevAj9zW2Ao1o-U15z0fia8mIHTPYP2KT_TnGQJ65ZOtlAJVlzyawCeikOO32wYZ5R11XI1PtR4y6dW_4sGSbo8Pn3bakYg66Pu2olgRtG79wOJCjtSbhFa696ty_dRxD17d-WfzMZ59T0CFbs1eU0_YBSUXCunYTGXBwF--RwGA6R6f5KQ8xNArklMqYqJAhskx8Z_AXJVIpHu3nMZVCySZIlYalO0sHTr_pLDxHCqSRzIKslQdsewNcZd6wuS3W-mTyrbDZ57IYx7y0C0

La elección de aplicar el patrón Factory Method se debe a que el problema es justamente la creación de objetos (en este caso, pedidos). Aunque también podría utilizarse un factory simple, resulta más apropiado usar este patrón porque permite que diferentes tipos de restaurantes o canales de atención definan cómo crear su propio pedido sobreescribiendo el método `create_order`.

De esta manera, es posible incorporar nuevos tipos de creadores sin necesidad de modificar la clase Restaurant, lo que facilita la extensión del sistema y reduce el acoplamiento.